# RAG Text Chunking Factory Demo

Explore the new YAML-configured chunking system with auto-detection and flexible strategies.

**What this notebook covers:**

1. **ChunkerFactory** — create chunkers from YAML config, auto-detect by file extension
2. **Chunking Strategies** — markdown (MarkdownChef), recursive, chonkie-recursive
3. **Metadata Extraction** — token counts, chunk types, positions
4. **Comparison** — how different chunkers split the same document
5. **Integration** — use with RAG retrieval and ingestion flows

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from dotenv import load_dotenv
from rich import print  # noqa: F401
from rich.table import Table
from rich.console import Console

assert load_dotenv(verbose=True)
sys.path.append(".")

console = Console()

## 1. Intro to ChunkerFactory

The `ChunkerFactory` provides two main methods:

- **`create(config_tag)`** — create a specific chunker from config
- **`create_for_file(path, chunker='auto')`** — auto-detect chunker by file extension

In [2]:
from genai_tk.extra.rag.chunker_factory import ChunkerFactory
from genai_tk.utils.config_mngr import global_config

# Load config
cfg = global_config()

# Inspect available chunkers
chunkers = cfg.get_dict("chunkers", {})
print(f"\n[bold]Available Chunkers:[/bold]")
for name in chunkers.keys():
    print(f"  - {name}")

# Inspect auto-detection map
auto_map = cfg.get_dict("chunker_auto_map", {})
print(f"\n[bold]File Extension Mappings:[/bold]")
table = Table(title="Auto-Detection Map")
table.add_column("Extension", style="cyan")
table.add_column("Chunker", style="green")
for ext, chunker in sorted(auto_map.items()):
    table.add_row(ext, chunker)
console.print(table)

Available Chunkers:

- recursive

- markdown

- chonkie_recursive

File Extension Mappings:

   Auto-Detection Map    
┏━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Extension ┃ Chunker   ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━┩
│ .default  │ recursive │
│ .java     │ recursive │
│ .js       │ recursive │
│ .markdown │ markdown  │
│ .md       │ markdown  │
│ .py       │ recursive │
│ .rst      │ markdown  │
│ .txt      │ recursive │
└───────────┴───────────┘

## 2. Creating Chunkers Explicitly

Create specific chunkers from configuration.

In [3]:
# Create different chunker types
recursive_splitter = ChunkerFactory.create("recursive")
markdown_splitter = ChunkerFactory.create("markdown")
chonkie_recursive_splitter = ChunkerFactory.create("chonkie_recursive")

print(f"[bold cyan]Recursive Splitter:[/bold cyan] {type(recursive_splitter).__name__}")
print(f"[bold cyan]Markdown Splitter:[/bold cyan] {type(markdown_splitter).__name__}")
print(f"[bold cyan]Chonkie Recursive Splitter:[/bold cyan] {type(chonkie_recursive_splitter).__name__}")

2026-05-06 09:47:38.955 | DEBUG    | genai_tk.extra.rag.chunker_factory:create:73 - Creating chunker 'recursive' from class langchain_text_splitters.RecursiveCharacterTextSplitter
2026-05-06 09:47:38.977 | DEBUG    | genai_tk.extra.rag.chunker_factory:create:73 - Creating chunker 'markdown' from class genai_tk.extra.rag.chonkie_splitter.ChonkieTextSplitter
2026-05-06 09:47:40.514 | DEBUG    | genai_tk.extra.rag.chunker_factory:create:73 - Creating chunker 'chonkie_recursive' from class genai_tk.extra.rag.chonkie_splitter.ChonkieTextSplitter


Recursive Splitter: RecursiveCharacterTextSplitter

Markdown Splitter: ChonkieTextSplitter

Chonkie Recursive Splitter: ChonkieTextSplitter

## 3. Auto-Detection by File Extension

Let the factory detect the right chunker based on file extension.

In [4]:
from upath import UPath

# Test auto-detection
test_files = [
    "document.md",
    "readme.markdown",
    "script.py",
    "app.js",
    "notes.txt",
    "unknown.xyz",
]

table = Table(title="Auto-Detection Results")
table.add_column("File", style="cyan")
table.add_column("Detected Chunker", style="green")
table.add_column("Splitter Type", style="yellow")

for file in test_files:
    path = UPath(file)
    splitter = ChunkerFactory.create_for_file(path, "auto")
    table.add_row(file, auto_map.get(path.suffix, auto_map.get(".default", "unknown")), type(splitter).__name__)

console.print(table)

2026-05-06 09:47:40.776 | DEBUG    | genai_tk.extra.rag.chunker_factory:create_for_file:118 - Auto-selected chunker 'markdown' for extension '.md'
2026-05-06 09:47:40.795 | DEBUG    | genai_tk.extra.rag.chunker_factory:create:73 - Creating chunker 'markdown' from class genai_tk.extra.rag.chonkie_splitter.ChonkieTextSplitter
2026-05-06 09:47:40.969 | DEBUG    | genai_tk.extra.rag.chunker_factory:create_for_file:118 - Auto-selected chunker 'markdown' for extension '.markdown'
2026-05-06 09:47:40.991 | DEBUG    | genai_tk.extra.rag.chunker_factory:create:73 - Creating chunker 'markdown' from class genai_tk.extra.rag.chonkie_splitter.ChonkieTextSplitter
2026-05-06 09:47:41.163 | DEBUG    | genai_tk.extra.rag.chunker_factory:create_for_file:118 - Auto-selected chunker 'recursive' for extension '.py'
2026-05-06 09:47:41.180 | DEBUG    | genai_tk.extra.rag.chunker_factory:create:73 - Creating chunker 'recursive' from class langchain_text_splitters.RecursiveCharacterTextSplitter
2026-05-06 09:

                        Auto-Detection Results                         
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ File            ┃ Detected Chunker ┃ Splitter Type                  ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ document.md     │ markdown         │ ChonkieTextSplitter            │
│ readme.markdown │ markdown         │ ChonkieTextSplitter            │
│ script.py       │ recursive        │ RecursiveCharacterTextSplitter │
│ app.js          │ recursive        │ RecursiveCharacterTextSplitter │
│ notes.txt       │ recursive        │ RecursiveCharacterTextSplitter │
│ unknown.xyz     │ recursive        │ RecursiveCharacterTextSplitter │
└─────────────────┴──────────────────┴────────────────────────────────┘

## 4. Comparing Chunking Strategies

See how different chunkers split the same markdown content.

In [5]:
# Sample markdown document
markdown_text = """
# Introduction to RAG Systems

Retrieval-Augmented Generation (RAG) combines document retrieval with large language models.

## How RAG Works

1. **Embedding** — Convert documents to embeddings and store in vector database
2. **Retrieval** — Find relevant documents for a query
3. **Generation** — Use retrieved docs as context for the LLM

## Benefits

- More accurate answers (grounded in actual documents)
- Reduced hallucinations
- Easy to update knowledge without retraining

## Document Chunking

Breaking documents into chunks is critical for RAG.
Different chunking strategies work better for different content types.

| Strategy | Best For |
|----------|----------|
| Markdown | Structured docs (markdown, RST) |
| Recursive | Code and plain text |
| Semantic | Long-form narrative |

```python
from genai_tk.extra.rag.chunker_factory import ChunkerFactory

splitter = ChunkerFactory.create("markdown")
docs = splitter.split_documents([document])
```
"""

print(f"\n[bold]Original text length:[/bold] {len(markdown_text)} characters")

Original text length: 961 characters

In [6]:
# Chunk with markdown splitter
markdown_splitter = ChunkerFactory.create("markdown")
markdown_docs = markdown_splitter.create_documents([markdown_text], metadatas=[{"source": "rag_intro.md"}])

print(f"\n[bold cyan]Markdown Chunker Results:[/bold cyan]")
table = Table(title="Markdown Chunks")
table.add_column("#", style="cyan")
table.add_column("Token Count", style="green")
table.add_column("Type", style="yellow")
table.add_column("Preview", style="white")

for i, doc in enumerate(markdown_docs, 1):
    token_count = doc.metadata.get("token_count", 0)
    chunk_type = doc.metadata.get("chunk_type", "text")
    preview = doc.page_content[:60].replace("\n", " ").strip()
    table.add_row(str(i), str(token_count), chunk_type, preview + "...")

console.print(table)

2026-05-06 09:47:41.604 | DEBUG    | genai_tk.extra.rag.chunker_factory:create:73 - Creating chunker 'markdown' from class genai_tk.extra.rag.chonkie_splitter.ChonkieTextSplitter


Markdown Chunker Results:

                                      Markdown Chunks                                       
┏━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Token Count ┃ Type  ┃ Preview                                                        ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ 204         │ table │ # Introduction to RAG Systems  Retrieval-Augmented Generati... │
└───┴─────────────┴───────┴────────────────────────────────────────────────────────────────┘

In [7]:
# Chunk with recursive splitter
recursive_splitter = ChunkerFactory.create("recursive")
recursive_docs = recursive_splitter.create_documents([markdown_text], metadatas=[{"source": "rag_intro.md"}])

print(f"\n[bold cyan]Recursive Chunker Results:[/bold cyan]")
table = Table(title="Recursive Chunks")
table.add_column("#", style="cyan")
table.add_column("Token Count", style="green")
table.add_column("Type", style="yellow")
table.add_column("Preview", style="white")

for i, doc in enumerate(recursive_docs, 1):
    token_count = doc.metadata.get("token_count", 0)
    chunk_type = doc.metadata.get("chunk_type", "text")
    preview = doc.page_content[:60].replace("\n", " ").strip()
    table.add_row(str(i), str(token_count), chunk_type, preview + "...")

console.print(table)

2026-05-06 09:47:41.862 | DEBUG    | genai_tk.extra.rag.chunker_factory:create:73 - Creating chunker 'recursive' from class langchain_text_splitters.RecursiveCharacterTextSplitter


Recursive Chunker Results:

                                      Recursive Chunks                                      
┏━━━┳━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Token Count ┃ Type ┃ Preview                                                         ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ 0           │ text │ # Introduction to RAG Systems  Retrieval-Augmented Generatio... │
│ 2 │ 0           │ text │ ## Document Chunking  Breaking documents into chunks is crit... │
└───┴─────────────┴──────┴─────────────────────────────────────────────────────────────────┘

## 5. Metadata in Documents

Each chunk carries rich metadata for tracking and debugging.

In [8]:
# Inspect metadata from first chunk
if markdown_docs:
    doc = markdown_docs[0]
    print(f"\n[bold cyan]Metadata in Document #1:[/bold cyan]")
    table = Table(title="Document Metadata")
    table.add_column("Field", style="cyan")
    table.add_column("Value", style="green")

    for key, value in sorted(doc.metadata.items()):
        table.add_row(key, str(value))

    console.print(table)

    print(f"\n[bold cyan]Content Preview:[/bold cyan]")
    print(doc.page_content[:200] + "...")

Metadata in Document #1:

      Document Metadata       
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Field       ┃ Value        ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ chunk_type  │ table        │
│ source      │ rag_intro.md │
│ start_index │ 0            │
│ token_count │ 204          │
└─────────────┴──────────────┘

Content Preview:

# Introduction to RAG Systems

Retrieval-Augmented Generation (RAG) combines document retrieval with large language models.

## How RAG Works

1. **Embedding** — Convert documents to embeddings and s...

## 6. Comparison Summary

Key differences between chunking strategies.

## 6b. Table Splitting — Columns Preserved Per Chunk

`MarkdownChef` extracts tables as dedicated chunks, keeping all columns intact.
Each table chunk gets `chunk_type="table"` in its metadata.

In [17]:
import re

# Document with multiple tables and prose
table_demo_text = """
# Product Catalogue

Below is an overview of available products.

| Product   | Category    | Price  | Stock |
|-----------|-------------|--------|-------|
| Widget A  | Electronics | $29.99 | 150   |
| Widget B  | Electronics | $49.99 | 80    |
| Gadget X  | Tools       | $15.00 | 500   |

Some explanatory text between tables.

## Regional Sales

| Region  | Q1     | Q2     | Q3     | Q4     |
|---------|--------|--------|--------|--------|
| North   | 1,200  | 1,500  | 1,100  | 1,800  |
| South   | 900    | 1,050  | 980    | 1,200  |
| East    | 1,400  | 1,350  | 1,600  | 1,750  |
| West    | 800    | 950    | 870    | 1,100  |

Totals are aggregated from internal reporting.
"""


def _parse_table_columns(text: str) -> list[str] | None:
    """Return column header names if the text contains a markdown table, else None."""
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("|") and "---" not in line and line.count("|") >= 2:
            cols = [c.strip() for c in line.strip("|").split("|") if c.strip()]
            if cols:
                return cols
    return None


splitter = ChunkerFactory.create("markdown")
table_docs = splitter.create_documents([table_demo_text], metadatas=[{"source": "catalogue.md"}])

print(f"\n[bold]Document split into {len(table_docs)} chunk(s)[/bold]\n")

tbl = Table(title="Chunks — Table Columns Preserved", show_lines=True)
tbl.add_column("#", style="cyan", width=3)
tbl.add_column("Type", style="yellow", width=8)
tbl.add_column("Tokens", style="green", width=7)
tbl.add_column("Columns (tables) / Preview (text)", style="white")

for i, doc in enumerate(table_docs, 1):
    chunk_type = doc.metadata.get("chunk_type", "text")
    tokens = doc.metadata.get("token_count", 0)

    # Try to extract table columns from the content regardless of chunk_type label
    cols = _parse_table_columns(doc.page_content)
    if cols is not None:
        detail = "[bold magenta]" + "  |  ".join(cols) + "[/bold magenta]"
        chunk_type = "table"  # override display label if columns were found
    else:
        detail = doc.page_content[:80].replace("\n", " ").strip() + "…"

    tbl.add_row(str(i), chunk_type, str(tokens), detail)

console.print(tbl)

# Summary
table_chunks = [d for d in table_docs if _parse_table_columns(d.page_content) is not None]
text_chunks  = [d for d in table_docs if _parse_table_columns(d.page_content) is None]
print(f"\n  Table chunks : [bold magenta]{len(table_chunks)}[/bold magenta]  (each contains all columns)")
print(f"  Text/prose chunks : [bold cyan]{len(text_chunks)}[/bold cyan]")


2026-05-06 10:21:53.303 | DEBUG    | genai_tk.extra.rag.chunker_factory:create:73 - Creating chunker 'markdown' from class genai_tk.extra.rag.chonkie_splitter.ChonkieTextSplitter


Document split into 2 chunk(s)

                   Chunks — Table Columns Preserved                    
┏━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Type     ┃ Tokens  ┃ Columns (tables) / Preview (text)        ┃
┡━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ table    │ 100     │ Product  |  Category  |  Price  |  Stock │
├─────┼──────────┼─────────┼──────────────────────────────────────────┤
│ 2   │ table    │ 133     │ Region  |  Q1  |  Q2  |  Q3  |  Q4       │
└─────┴──────────┴─────────┴──────────────────────────────────────────┘

Table chunks : 2  (each contains all columns)

Text/prose chunks : 0

In [9]:
# Compare statistics
total_tokens_markdown = sum(d.metadata.get("token_count", 0) for d in markdown_docs)
total_tokens_recursive = sum(d.metadata.get("token_count", 0) for d in recursive_docs)

print(f"\n[bold]Chunking Strategy Comparison:[/bold]")
table = Table(title="Statistics")
table.add_column("Metric", style="cyan")
table.add_column("Markdown", style="green")
table.add_column("Recursive", style="yellow")

table.add_row("# Chunks", str(len(markdown_docs)), str(len(recursive_docs)))
table.add_row("Total Tokens", str(total_tokens_markdown), str(total_tokens_recursive))
avg_md = total_tokens_markdown / len(markdown_docs) if markdown_docs else 0
avg_rec = total_tokens_recursive / len(recursive_docs) if recursive_docs else 0
table.add_row("Avg Tokens/Chunk", f"{avg_md:.1f}", f"{avg_rec:.1f}")

console.print(table)

Chunking Strategy Comparison:

                Statistics                 
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Metric           ┃ Markdown ┃ Recursive ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━┩
│ # Chunks         │ 1        │ 2         │
│ Total Tokens     │ 204      │ 0         │
│ Avg Tokens/Chunk │ 204.0    │ 0.0       │
└──────────────────┴──────────┴───────────┘

## 7. Custom Chunker Configuration

You can customize chunker parameters in `config/rag.yaml` or create your own.

In [12]:
# Create a custom markdown splitter with different parameters
from genai_tk.extra.rag.chonkie_splitter import ChonkieTextSplitter

# Smaller chunks for more granularity
small_chunk_splitter = ChonkieTextSplitter(
    chunker_type="markdown",
    max_tokens=200,  # smaller chunks
    min_tokens=25,  # merge tiny chunks
    merge_small_chunks=True,
    encoding_name="o200k_base",
)

small_docs = small_chunk_splitter.create_documents(
    [markdown_text], metadatas=[{"source": "rag_intro.md", "chunk_size": "small"}]
)

print(f"\n[bold cyan]Custom Splitter (max_tokens=200):[/bold cyan]")
print(f"  Chunks: {len(small_docs)}")
print(f"  Avg tokens: {sum(d.metadata.get('token_count', 0) for d in small_docs) / len(small_docs):.1f}")

Custom Splitter (max_tokens=200):

Chunks: 3

Avg tokens: 67.7

## 8. Integration with RAG Pipeline

Use the chunker factory with file ingestion and retrieval.

In [13]:
# Example: Ingest markdown files with auto-detection
print(f"\n[bold cyan]CLI Usage Examples:[/bold cyan]")
print("""
# Auto-detect chunker by file extension
uv run cli rag add-files ./docs --chunker auto

# Use explicit chunker
uv run cli rag add-files ./docs --chunker markdown

# Adjust chunk size
uv run cli rag add-files ./docs --chunker auto --chunk-size 256

# Specify retriever
uv run cli rag add-files ./docs --chunker auto --retriever persistent
""")

CLI Usage Examples:

# Auto-detect chunker by file extension
uv run cli rag add-files ./docs --chunker auto

# Use explicit chunker
uv run cli rag add-files ./docs --chunker markdown

# Adjust chunk size
uv run cli rag add-files ./docs --chunker auto --chunk-size 256

# Specify retriever
uv run cli rag add-files ./docs --chunker auto --retriever persistent

## 9. Programmatic Ingestion

Use the Prefect flow for batch document processing with chunker selection.

In [14]:
from genai_tk.extra.prefect.runtime import run_flow_ephemeral
from genai_tk.extra.rag.rag_prefect_flow import rag_file_ingestion_flow
import tempfile
from pathlib import Path

# Create a temp directory with sample files
with tempfile.TemporaryDirectory() as tmpdir:
    tmppath = Path(tmpdir)

    # Create sample markdown file
    (tmppath / "guide.md").write_text(
        "# Quick Start\n\nFollow these steps to get started.\n\n## Step 1\n\nDo this first."
    )

    # Create sample text file
    (tmppath / "notes.txt").write_text(
        "These are important notes.\n\nNote 1: Something important.\n\nNote 2: Another note."
    )

    print(f"[bold cyan]Ingesting files from:[/bold cyan] {tmppath}")

    # Ingest with auto-detection
    result = run_flow_ephemeral(
        rag_file_ingestion_flow,
        root_dir=str(tmppath),
        retriever_name="default",
        max_chunk_tokens=512,
        chunker_name="auto",  # Auto-detect by extension
        include_patterns=["**/*.md", "**/*.txt"],
        force=True,
    )

    print(f"\n[bold cyan]Ingestion Results:[/bold cyan]")
    table = Table()
    table.add_column("Metric", style="cyan")
    table.add_column("Value", style="green")
    table.add_row("Total Files", str(result.get("total_files", 0)))
    table.add_row("Processed", str(result.get("processed_files", 0)))
    table.add_row("Skipped", str(result.get("skipped_files", 0)))
    table.add_row("Total Chunks", str(result.get("total_chunks", 0)))
    console.print(table)

Ingesting files from: /tmp/tmpjtz5n6xw

2026-05-06 10:11:43.016 | INFO     | genai_tk.extra.rag.rag_prefect_flow:rag_file_ingestion_flow:194 - Starting RAG file ingestion from '/tmp/tmpjtz5n6xw' to retriever 'default'
2026-05-06 10:11:43.027 | INFO     | genai_tk.extra.rag.rag_prefect_flow:rag_file_ingestion_flow:210 - Found 2 files matching patterns
2026-05-06 10:11:43.985 | DEBUG    | genai_tk.core.embeddings_store:get_vector_store:237 - Created vector store: genai_tk.core.vector_backends.ChromaBackend/embeddings_qwen3_06b => 'in-memory'
2026-05-06 10:11:43.987 | INFO     | genai_tk.extra.rag.rag_prefect_flow:rag_file_ingestion_flow:218 - Processing 2 files, skipping 0 already processed files
2026-05-06 10:11:43.988 | INFO     | genai_tk.extra.rag.rag_prefect_flow:rag_file_ingestion_flow:226 - Processing batch 1 (2 files)
2026-05-06 10:11:44.021 | INFO     | genai_tk.extra.rag.rag_prefect_flow:process_file_task:120 - Processing file: /tmp/tmpjtz5n6xw/notes.txt (chunker: auto)
2026-05-06 10:11:44.023 | INFO     | genai_tk.e

Ingestion Results:

┏━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Metric       ┃ Value ┃
┡━━━━━━━━━━━━━━╇━━━━━━━┩
│ Total Files  │ 2     │
│ Processed    │ 2     │
│ Skipped      │ 0     │
│ Total Chunks │ 2     │
└──────────────┴───────┘

## 10. Key Takeaways

- **ChunkerFactory** — Unified factory for all chunking strategies
- **Auto-Detection** — Select chunker by file extension automatically
- **Rich Metadata** — Track token counts, types, and positions
- **Flexible Config** — Define chunkers in `config/rag.yaml`
- **Integration** — Works seamlessly with CLI, Prefect flows, and agents